# Real-project showcase: Forsthausgasse

Compare two stored Forsthausgasse variants using the same schematic building layout.
The scenario areas, glazing and energy-system inputs come directly from the project
used in the Excel how-tos. The two-building arrangement uses the requested 2:1 length
ratio, with an assumed depth of 15 m.

This example also illustrates why drawings should follow populated inputs rather
than scenario names: the variant labelled `WP` still contains an active district-heating
slot in this export. We show that discrepancy explicitly.

In [ ]:
from pathlib import Path
from copy import deepcopy
from html import escape
from textwrap import wrap
from xml.etree import ElementTree as ET
import math
import pandas as pd
from IPython.display import display, HTML
from pexl import Project
from pexl.spatial import section_layout_from_scenario, render_svg

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_PATH = ROOT / "data/exports/ka_project_backup_v1_13.xlsx"
project = Project.from_excel(PROJECT_PATH)
names = ["Forsthausgasse | BT2 KON FW", "Forsthausgasse | BT2 ÖKO WP FLEX 2,6xPV"]
scenarios = [next(s for s in project.scenarios if s.column_name == name) for name in names]

def number(scenario, name):
    try:
        value = float(getattr(scenario.v, name, None))
        return value if math.isfinite(value) else None
    except (TypeError, ValueError):
        return None

def show_svg(svg):
    ET.fromstring(svg)
    display(HTML('<div style="overflow:auto">'+svg+'</div>'))


## 1. Inspect the source

Both variants contain approximately 5,510 m² of residential NFA. The baseline has
no PV module area; the second variant has 185 m² and 37 kWp. The recorded building
count is 70, whereas this showcase deliberately uses the two-building arrangement
specified for the section. The input is not overwritten.

A non-zero COP or borehole count does not by itself establish an active heat pump.
Check the generator capacities before interpreting the carrier or source fields.

In [ ]:
fields = ["NFA_total", "NFA_residential", "building_count", "storey_count_avg",
          "hull_fenestration_rate", "PV_module_area", "PV_kWp",
          "QHmax_1el_m2", "heat_cop_1el", "QHmax_2th_m2", "heat_th2_carrier_type",
          "QHmax_3el_m2", "QHmax_4th_m2", "borehole_count",
          "vent_share_mechanical_residential", "vent_heat_recovery_winter_residential",
          "preset_recorded_heating_system", "preset_recorded_heating_emission_system"]
source_values = pd.DataFrame({s.column_name: [getattr(s.v, key, None) for key in fields]
                             for s in scenarios}, index=fields)
display(source_values)


## 2. Use the same layout for both variants

The maximum number of above-ground floors is the rounded-up average from the
baseline. The layout generator conserves each scenario's NFA and derives building
lengths from the capacities, depth and 2:1 ratio. Partial top floors are allowed.
No basement is added because its geometry is unknown.

The overlay includes only supported, evidenced components: an active district-heat
connection and mechanical ventilation. Room-side heating and cooling emitters remain
unspecified. Plant assignment to individual buildings is not recorded; the shared
overlay demonstrates the same service in both buildings, not two verified plant rooms.
The lift, stairs, street and courtyard are schematic renderer context.

In [ ]:
layout_base = {"buildings": [
    {"name": "Building 1", "relative_length": 2, "depth_m": 15,
     "max_floors": math.ceil(number(scenarios[0], "storey_count_avg")), "usages": ["residential"]},
    {"name": "Building 2", "relative_length": 1, "depth_m": 15,
     "max_floors": math.ceil(number(scenarios[0], "storey_count_avg")), "usages": ["residential"]},
]}

def evidenced_systems(scenario):
    plant = []
    for slot in (2, 4):
        if (number(scenario, f"QHmax_{slot}th_m2") or 0) <= 0:
            continue
        carrier = str(getattr(scenario.v, f"heat_th{slot}_carrier_type", ""))
        if "Fernw" in carrier: plant.append("district_heating")
        elif carrier == "Erdgas": plant.append("gas_boiler")
        elif "Biomasse" in carrier: plant.append("biomass_boiler")
    ventilation = "none"
    if number(scenario, "vent_share_mechanical_residential") == 1:
        ventilation = "heat_recovery" if (number(scenario, "vent_heat_recovery_winter_residential") or 0)>0 else "mechanical"
    return {"plant": sorted(set(plant)), "roof": ["pv"],
            "usages": {"residential": {"ventilation": ventilation}}}

sections, svgs = [], []
for scenario in scenarios:
    layout = deepcopy(layout_base)
    layout["systems"] = evidenced_systems(scenario)
    section = section_layout_from_scenario(scenario, layout=layout, seed=42)
    sections.append(section)
    svgs.append(render_svg(section, style="architectural"))


## 3. Compare the real variants

The comparison uses a common crop and scale, so the PV difference is visible without
changing the building geometry. Full-size sections with their legends follow below.
The captions report stored inputs, not recalculated performance.

In [ ]:
NS = "{http://www.w3.org/2000/svg}"
roots = [ET.fromstring(svg) for svg in svgs]
roof_y = [float(e.get("y")) for root in roots for e in root.iter()
          if e.get("data-kind") == "roof"]
legend_y = [float(e.get("y")) for root in roots for e in root.iter()
            if e.tag == NS+"text" and e.text == "Usage"]
crop_top = min(roof_y)-70
crop_bottom = min(legend_y)-35
crop_width = max(float(root.get("width")) for root in roots)
crop_height = crop_bottom-crop_top

board = ET.Element("svg", xmlns="http://www.w3.org/2000/svg", width="1440",
                   height="850", viewBox="0 0 1440 850", role="img")
ET.SubElement(board,"title").text = "Forsthausgasse: two real exported variants"
ET.SubElement(board,"rect",width="1440",height="850",fill="#f5f6f4")
def text(value,x,y,size=14,**attrs):
    ET.SubElement(board,"text",dict(x=str(x),y=str(y),fill="#40545e",
        **{"font-family":"Arial, sans-serif","font-size":str(size)},**attrs)).text=value
text("FORSTHAUSGASSE",26,39,26)
text("Real exported inputs / shared two-building schematic / 2:1 length ratio",26,67,15)
for i,(scenario,root) in enumerate(zip(scenarios,roots)):
    left=20+i*710
    ET.SubElement(board,"rect",x=str(left),y="91",width="690",height="640",rx="8",fill="white",stroke="#d4dddf")
    for line,value in enumerate(wrap(scenario.column_name,48)):
        text(value,left+18,118+line*20,16)
    text(f"NFA {number(scenario,'NFA_total'):,.0f} m² | PV {number(scenario,'PV_module_area'):g} m² / {number(scenario,'PV_kWp'):g} kWp",left+18,170,14)
    nested=deepcopy(root)
    for node in nested.iter():
        node.tag = node.tag.rsplit("}", 1)[-1]  # Inherit the outer SVG namespace in HTML.
    nested.set("x",str(left+8));nested.set("y","191");nested.set("width","674");nested.set("height",str(674*crop_height/crop_width)); nested.set("overflow","hidden")
    nested.set("viewBox",f"0 {crop_top} {crop_width} {crop_height}")
    # Prefix IDs to keep the two complete SVG trees independent.
    ids={e.get("id"):f"case{i}_{e.get('id')}" for e in nested.iter() if e.get("id")}
    for e in nested.iter():
        if e.get("id"):e.set("id",ids[e.get("id")])
        for key,value in list(e.attrib.items()):
            for old,new in ids.items():value=value.replace(f"url(#{old})",f"url(#{new})")
            e.set(key,value)
    board.append(nested)
    text("Drawn: active district heating; ventilation follows each variant",left+18,636,13)
    text("Emitter type and plant locations: not recorded",left+18,660,13)
    text("Electric heat capacity: 0 W/m² in the export",left+18,684,13)
text("Read the values, not just the variant title",26,772,17)
text("The WP-labelled variant still contains active district heating. Its heat-pump source cannot be reconstructed from this export.",26,800,13)
text("Layout override: 2 buildings, depth 15 m, max. 4 floors. Source areas and energy inputs are unchanged.",26,824,13)
showcase_svg = ET.tostring(board,encoding="unicode")
show_svg(showcase_svg)


## 4. Inspect each section at full size

The legend identifies usage, material and HVAC overlays. The façade glazing is
based on the scenario's fenestration ratio. The lower annotations report PV area
and any capacity that could not be placed.

In [ ]:
for scenario, svg in zip(scenarios, svgs):
    display(HTML("<h3>"+escape(scenario.column_name)+"</h3>"))
    show_svg(svg)

## 5. What this showcase establishes

- The two variants share the same residential area and the same section layout.
- The PV area differs: 0 versus 185 m², corresponding to 0 versus 37 kWp.
- Both populated heating configurations select district heating; the `WP` title is
  not sufficient evidence for drawing a heat pump.
- Recorded emission preset names and building-specific equipment locations are absent.

Once these details are confirmed, add them explicitly to each building's `systems`
dictionary. The [HVAC gallery](howto_hvac_gallery.ipynb) shows the available and
proposed component representations.

## Technical notes and export

The source workbook is loaded through `Project.from_excel`. No values are written
back to it. Area conservation and preservation of the inspected inputs are checked
below. The optional export cell saves the comparison and the individual sections.

In [ ]:
for scenario, section in zip(scenarios, sections):
    assert math.isclose(sum(s["area_m2"] for s in section["spaces"]), number(scenario,"NFA_total"), rel_tol=1e-9)
    assert len(section["buildings"]) == 2
    assert not any("water_well" in v["plant"] or "boreholes" in v["plant"]
                   for v in section["systems"]["buildings"].values())
    ET.fromstring(render_svg(section))
after = pd.DataFrame({s.column_name:[getattr(s.v,key,None) for key in fields] for s in scenarios},index=fields)
pd.testing.assert_frame_equal(source_values,after)
print("Validated: two real scenarios, conserved NFA, unchanged source inputs, valid SVG.")

EXPORT_DIR = None  # Example: Path("forsthausgasse_showcase")
if EXPORT_DIR is not None:
    EXPORT_DIR.mkdir(parents=True,exist_ok=True)
    (EXPORT_DIR/"comparison.svg").write_text(showcase_svg,encoding="utf-8")
    for i,svg in enumerate(svgs,1):
        (EXPORT_DIR/f"variant_{i}.svg").write_text(svg,encoding="utf-8")
